In [ ]:
# stock_forecast_app.py
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.seasonal import STL
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# -----------------------------------------------
# Streamlit UI
# -----------------------------------------------
st.set_page_config(page_title="Stock Price Forecasting (LSTM + STL)", layout="wide")

st.title("📈 Stock Price Forecast using LSTM & STL Decomposition")

st.sidebar.header("Input Parameters")

ticker = st.sidebar.text_input("Enter Stock Ticker (e.g., HDFCBANK.NS)", "HDFCBANK.NS")
start_date = st.sidebar.date_input("Start Date", datetime(2020, 1, 1))
end_date = st.sidebar.date_input("End Date", datetime(2025, 8, 6))
epochs = st.sidebar.slider("Training Epochs", 5, 50, 10)
window_size = st.sidebar.slider("LSTM Window Size", 5, 30, 10)

if st.sidebar.button("Run Forecast"):
    st.info(f"Fetching data for **{ticker}** from {start_date} to {end_date}...")
    data = yf.download(ticker, start=start_date, end=end_date)

    if data.empty:
        st.error("No data found for this ticker and date range.")
        st.stop()

    d_high = data["High"]

    st.subheader("📊 Raw Stock Data (Last 5 Rows)")
    st.write(data.tail())

    # -----------------------------------------------
    # STL Decomposition
    # -----------------------------------------------
    st.info("Performing STL Decomposition...")
    stl = STL(d_high, period=30)
    result = stl.fit()
    trend = result.trend
    seasonal = result.seasonal
    resid = result.resid

    fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    ax[0].plot(trend, label="Trend")
    ax[1].plot(seasonal, label="Seasonal", color='orange')
    ax[2].plot(resid, label="Residual", color='green')
    for a in ax:
        a.legend(); a.grid(True)
    st.pyplot(fig)

    # -----------------------------------------------
    # Data Preparation
    # -----------------------------------------------
    def prepare_lstm_data(series, window_size):
        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(series.values.reshape(-1, 1))
        X, y = [], []
        for i in range(len(scaled) - window_size):
            X.append(scaled[i:i + window_size])
            y.append(scaled[i + window_size])
        return np.array(X), np.array(y), scaler

    X_trend, y_trend, scaler_trend = prepare_lstm_data(trend, window_size)
    X_seasonal, y_seasonal, scaler_seasonal = prepare_lstm_data(seasonal, window_size)

    # -----------------------------------------------
    # Build and Train LSTM
    # -----------------------------------------------
    @st.cache_resource
    def build_and_train_lstm(X, y, epochs):
        model = Sequential([
            LSTM(50, return_sequences=True, input_shape=(X.shape[1], 1)),
            LSTM(50, return_sequences=True),
            LSTM(50, return_sequences=True),
            LSTM(50, return_sequences=False),
            Dense(1)
        ])
        model.compile(optimizer='adam', loss='mse')
        model.fit(X, y, epochs=epochs, batch_size=10, verbose=0)
        return model

    st.info("Training LSTM models (Trend & Seasonal)... ⏳")
    model_trend = build_and_train_lstm(X_trend, y_trend, epochs)
    model_seasonal = build_and_train_lstm(X_seasonal, y_seasonal, epochs)

    # -----------------------------------------------
    # Predictions
    # -----------------------------------------------
    y_trend_pred = model_trend.predict(X_trend)
    y_seasonal_pred = model_seasonal.predict(X_seasonal)

    trend_pred = scaler_trend.inverse_transform(y_trend_pred)
    seasonal_pred = scaler_seasonal.inverse_transform(y_seasonal_pred)

    final_pred = trend_pred.flatten() + seasonal_pred.flatten()
    actual = d_high.values[window_size:]

    # -----------------------------------------------
    # Plot Actual vs Predicted
    # -----------------------------------------------
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    ax2.plot(actual, label="Actual", color='blue')
    ax2.plot(final_pred, label="Predicted (Trend + Seasonal)", color='red')
    ax2.set_title("LSTM Forecast on STL Components")
    ax2.legend(); ax2.grid(True)
    st.pyplot(fig2)

    # -----------------------------------------------
    # RMSE
    # -----------------------------------------------
    rmse = np.sqrt(mean_squared_error(actual, final_pred))
    st.success(f"📉 Final RMSE (Reconstructed): {rmse:.4f}")

    # -----------------------------------------------
    # Next Day Forecast
    # -----------------------------------------------
    st.info("Generating Next-Day Forecast...")

    # Trend
    last_trend_scaled = scaler_trend.transform(trend.values[-window_size:].reshape(-1, 1)).reshape(1, window_size, 1)
    next_trend_scaled = model_trend.predict(last_trend_scaled)
    next_trend = scaler_trend.inverse_transform(next_trend_scaled)[0][0]

    # Seasonal
    last_seasonal_scaled = scaler_seasonal.transform(seasonal.values[-window_size:].reshape(-1, 1)).reshape(1, window_size, 1)
    next_seasonal_scaled = model_seasonal.predict(last_seasonal_scaled)
    next_seasonal = scaler_seasonal.inverse_transform(next_seasonal_scaled)[0][0]

    next_day_forecast = next_trend + next_seasonal

    st.subheader("📅 Forecast for Next Day")
    st.metric(label="Predicted High Price", value=f"{next_day_forecast:.2f}")

    st.balloons()